## 17.3. Q-Learning

### 17.3.1. The Q-Learning Algorithm

$$
Q_{k+1}(s, a) = r(s, a) + γ\sum_{s' \in \mathcal{S}}P(s' | s, a)\max_{a'∈\mathcal{A}}Q_k(s', a'); ∀ s ∈ \mathcal{S}, a∈\mathcal{A}
$$

### 17.3.2. An Optimization Problem Underlying Q-Learning

$$
\hat{Q} = \min_{Q}\frac1{nT}\sum_{i=1}^n\sum_{t=0}^{T-1}(Q(s_t^i, a_t^i) - r(s_t^i, a_t^i) - γ\max_{a'}Q(s_{t+1}^i, a'))^2 \stackrel{def}{=} \mathcal{\ell}(Q)
$$

$$
\begin{aligned}
Q(s_t^i, a_t^i) &\gets Q(s_t^i, a_t^i) - α∇_{Q(s^i_t, a_t^i)}ℓ(Q) \\ 
&=(1-\alpha)Q(s_t^i, a_t^i) - α(r(s_t^i,a_t^i) + γ\max_{a'}Q(s_{t+1}^i, a'))
\end{aligned}
$$

$$
Q(s_t^i, a_t^i) =(1-\alpha)Q(s_t^i, a_t^i) - α(r(s_t^i,a_t^i) + γ(1 - \mathbb{1}_{s_{t+1}^i \text{is terminal}})max_{a'}Q(s_{t+1}^i, a'))
$$

$$
\hat{π}(s) = \argmax_{a} \hat{Q}(s, a)
$$

### 17.3.3. Exploration in Q-Learning

ϵ-greedy exploration policy
$$
π_e(a | s) = \left\{\begin{aligned}
\argmax_{a'}\hat{Q}(s, a') &\text{ with prob.} 1 - ϵ \\ 
\text{uniform}(\mathcal{A}) &\text{ with prob.} ϵ
\end{aligned}\right.
$$

softmax exploration policy
$$
π_e(a | s) = \frac{\exp(\hat{Q}(s, a) / T)}{\sum_{a'}\exp{\hat{Q}(s, a') / T}}
$$

### 17.3.4. The "Self-correcting" Property of Q-Learning

### 17.3.5. Implementation of Q-Learning

In [ ]:
%matplotlib inline
import random
import numpy as np
from d2l import torch as d2l

seed = 0  # Random number generator seed
gamma = 0.95  # Discount factor
num_iters = 256  # Number of iterations
alpha   = 0.9  # Learing rate
epsilon = 0.9  # Epsilon in epsilion gready algorithm
random.seed(seed)  # Set the random seed
np.random.seed(seed)

# Now set up the environment
env_info = d2l.make_env('FrozenLake-v1', seed=seed)

: 

In [ ]:
def e_greedy(env, Q, s, epsilon):
  if random.random() < epsilon:
    return env.action_space.sample()
  else:
    return np.argmax(Q[s,:])

In [ ]:
def q_learning(env_info, gamma, num_iters, alpha, epsilon):
  env_desc = env_info['desc'] # 2d arry specifying what ecah grid item means
  env = env_info['env'] # 2d array specifying what each grid item means
  num_states = env_info['num_states']
  num_actions = env_info['num_actions']
  
  Q = np.zeros((num_states, num_actions))
  V = np.zeros((num_iters + 1, num_states))
  pi = np.zeros((num_iters + 1, num_states))
  
  